In [18]:
import torch
import torch.nn as nn

import sys
import os
import gc
import random
import optuna
import tqdm
import numpy as np
import pandas as pd

from torchvision import transforms
from astropy.io import fits
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt

parent_dir = os.path.abspath(os.path.join(os.path.dirname('utils.py'), ".."))
sys.path.append(parent_dir)
import utils as ut

In [19]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_FEATURES = ['FLAG']
#BANDS = ['f150w', 'f277w', 'f444w']
BANDS = ['f444w-f150w', 'f444w-f277w']
SEED = 42
BEST_ACC = 0

# HYPERPARAMETERS
INPUT_SIZE = 67
EPOCHS = 200
NUM_TRIALS = 20
PATIENCE = 20
OPTIMIZE = True

if OPTIMIZE:
    COMMENT = input("Enter OPTIMIZATION comment: ")

In [20]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

if SEED:
    ut.set_seeds(SEED)

## Prepare Data

In [21]:
cwd = Path.cwd()
cutout_dir = cwd.parent / 'data' / 'cutouts'
model_dir = cwd.parent / 'models'

In [22]:
# LOAD DATA INTO DF 

cols = {'idx': [],
        'images': [],
        'header': []}

for folder in cutout_dir.iterdir():
    if folder.is_dir():
        cols['idx'].append(folder.stem)

        temp_df = ut.fits_df(folder)
        temp_df.sort_values(by='BAND')

        # load images
        temp_imgs = dict()
        for _, row in temp_df.iterrows():
            hdul = fits.open(row['FILEPATH'])
            data = hdul[0].data.astype('float32')
            header = hdul[0].header
            temp_imgs[row['BAND']] = data
            hdul.close()
        cols['images'].append(temp_imgs)
        cols['header'].append(header)

# turn dict into df and expand header columns
df = pd.DataFrame(cols)
to_expand = ['header', 'images']
for col in to_expand:
    temp_df = pd.DataFrame(df[col].tolist(), index=df.index)
    df = df.join(temp_df)
df.drop(columns=['header', 'BAND', 'images'], inplace=True)

# shuffle
if SEED:
    df = df.sample(frac=1, random_state=SEED)
else:
    df = df.sample(frac=1)

In [ ]:
# GET SUBTRACTED AND DIVIDED BANDS

numers = ['f150w', 'f277w']
denominator = 'f444w'

for numerator in numers:
    df[numerator + '/' + denominator] = df[numerator] / df[denominator]
    df[denominator + '-' + numerator] = df[denominator] - df[numerator]

,idx,SIMPLE,BITPIX,NAXIS,NAXIS1,NAXIS2,RA,DEC,EXCESS,FLAG,SURVEY,TABLE,f150w,f277w,f444w,f150w/f444w,f444w-f150w,f277w/f444w,f444w-f277w
420,477,True,-32,2,50,50,34.450013,-5.169491,0.288462,0,uds,train_val_5,"[[-0.06858385, -0.019044295, 0.007521481, 0.04...","[[0.009278964, -0.0122975735, -0.012613724, 0....","[[0.0012348737, 0.0067816107, 0.0043256064, 0....","[[-55.539165, -2.8082259, 1.7388269, 4.6428127...","[[0.06981873, 0.025825907, -0.0031958744, -0.0...","[[7.5141, -1.8133706, -2.916059, 3.417338, -1....","[[-0.00804409, 0.019079184, 0.01693933, -0.022..."
262,334,True,-32,2,50,50,34.310911,-5.269744,2.795031,0,uds,train_val_8,"[[0.0061511355, -0.05197606, -0.06637941, 0.02...","[[0.027643101, 0.025135873, 0.025023365, 0.009...","[[0.009588264, 0.00910777, -0.050661813, -0.06...","[[0.64152753, -5.706782, 1.3102455, -0.4086601...","[[0.0034371284, 0.06108383, 0.0157176, -0.0943...","[[2.8830142, 2.7598271, -0.49392954, -0.143288...","[[-0.018054837, -0.016028102, -0.07568518, -0...."
431,5,True,-32,2,67,67,215.021537,52.991298,6.829268,0,ceers,train_val_6,"[[0.01921461, 0.031282034, 0.026716955, 0.0252...","[[0.015221625, 0.015132649, 0.017343683, 0.016...","[[0.018324139, 0.020127561, 0.017486572, 0.015...","[[1.0485955, 1.554189, 1.5278555, 1.6449069, 1...","[[-0.00089047104, -0.011154473, -0.009230383, ...","[[0.83068705, 0.7518372, 0.9918286, 1.0908843,...","[[0.003102514, 0.004994912, 0.0001428891, -0.0..."
448,65,True,-32,2,67,67,215.072598,52.933231,15.923567,1,ceers,train_val_7,"[[-0.0011979085, -0.0019694585, -0.0046428824,...","[[0.001100773, -0.0021963376, -0.006065462, -0...","[[-0.0045413487, -0.0044036983, -0.000549161, ...","[[0.26377815, 0.4472283, 8.454501, -1.4590477,...","[[-0.0033434401, -0.0024342397, 0.0040937215, ...","[[-0.24238901, 0.49874842, 11.044961, 2.203419...","[[-0.0056421217, -0.0022073607, 0.005516301, 0..."
364,426,True,-32,2,50,50,34.422590,-5.226552,3.954802,0,uds,train_val_8,"[[0.009081105, 0.074167006, 0.08961573, 0.0754...","[[-0.010331623, 0.019800717, 0.016585065, 0.00...","[[0.025066504, 0.00010402592, -0.009443422, 0....","[[0.36228046, 712.96655, -9.489752, 9.755367, ...","[[0.0159854, -0.07406298, -0.09905916, -0.0677...","[[-0.4121685, 190.34407, -1.7562559, 1.0837021...","[[0.035398126, -0.019696692, -0.026028488, -0...."


In [24]:
# STACK AND RESIZE BANDS

resize_transform = transforms.Resize(INPUT_SIZE,
                                     transforms.InterpolationMode.BILINEAR)

df['images'] = df[BANDS].values.tolist()
df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]

In [25]:
# SPLIT INTO TRAIN VAL TEST DFS

train_val_df = df[df['TABLE'] != 'test']
tables = sorted(list(train_val_df['TABLE'].unique()))

train_df = train_val_df[train_val_df["TABLE"].isin(tables[0:-2])]
val_df = train_val_df[~train_val_df["TABLE"].isin(tables[0:-2])]
test_df = df[df["TABLE"] == 'test']

len(train_df), len(val_df), len(test_df)

(342, 96, 48)

In [26]:
# EXTRACT X and y from dfs

X_train = torch.stack(train_df["images"].tolist())
X_val = torch.stack(val_df["images"].tolist())
X_test = torch.stack(test_df["images"].tolist())

y_train = torch.tensor(train_df["FLAG"].tolist())
y_val = torch.tensor(val_df["FLAG"].tolist())
y_test = torch.tensor(test_df["FLAG"].tolist())

i_train = torch.tensor(train_df["idx"].astype(int).tolist())
i_val = torch.tensor(val_df["idx"].astype(int).tolist())
i_test = torch.tensor(test_df["idx"].astype(int).tolist())

print(X_train.max())

tensor(64.9988)


In [27]:
# SCALE DATA

X_train.log1p_()
X_val.log1p_()
X_test.log1p_()

# 2. Calculate statistics on the newly modified log data
train_means = X_train.mean(axis=(0,2,3))
train_stds = X_train.std(axis=(0,2,3))

datasets = {
    'train': X_train,
    'val': X_val,
    'test': X_test
}

# Generate and print tables for each split
for name, data in datasets.items():
    means = data.mean(axis=(0, 2, 3))
    stds = data.std(axis=(0, 2, 3))
    maxes = float(data.max())
    
    df = pd.DataFrame({
        f'{name} mean': means,
        f'{name} std': stds,
        f'{name} max': maxes
    })
    
    print('\n', df.round(4))



    train mean  train std  train max
0         NaN        NaN        NaN
1         NaN        NaN        NaN

    val mean  val std  val max
0       NaN      NaN      NaN
1       NaN      NaN      NaN

    test mean  test std  test max
0        NaN       NaN       NaN
1        NaN       NaN       NaN


In [28]:
# CONVERT TO DATSETS

# transforms to apply to inputs
train_transform = transforms.Compose([ 
    #transforms.RandomHorizontalFlip(p=0.5),
    #transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=(0, 360)),
    #transforms.RandomPerspective()
    ])

train_set = ut.GalaxyDataset(images=X_train, ids=i_train,
                             labels=y_train, transform=train_transform)
val_set = ut.GalaxyDataset(images=X_val, ids=i_val ,labels=y_val)
test_set = ut.GalaxyDataset(images=X_test, ids=i_test ,labels=y_test)

In [29]:
# check sizes
print('train: ', train_set.__len__(),
      '\nval: ',val_set.__len__(),
      '\ntest: ',test_set.__len__(),
      '\ntotal: ', train_set.__len__() + val_set.__len__() + test_set.__len__())

train:  342 
val:  96 
test:  48 
total:  486


## Define Model

In [30]:
def define_model(trial):
    # get model structure from optuna
    n_conv = trial.suggest_int("n_conv", 1, 3)
    n_fc = trial.suggest_int("n_fc", 1, 3)
   
    conv_channels = [trial.suggest_int(f"n_channels_conv{i}", 1, 32) for i in range(3)]
    conv_ksizes = [trial.suggest_int(f"kernel_size_conv{i}", 1, 3) for i in range(3)]
    conv_paddings = [trial.suggest_int(f"padding_conv{i}", 1, 3) for i in range(3)]
    conv_dropouts = [trial.suggest_float(f"dropout_conv{i}", 0.2, 0.5) for i in range(3)]

    fc_units = [trial.suggest_int(f"n_units_fc{i}", 4, 256, step=4) for i in range(3)]
    fc_dropouts = [trial.suggest_float(f"dropout_fc{i}", 0.2, 0.5) for i in range(3)]

    layers = []

    # build conv layers
    in_channels = len(BANDS)
    current_spatial_size = INPUT_SIZE 

    for i in range(n_conv):
        out_channels = conv_channels[i]
        ksize = conv_ksizes[i]
        pad = conv_paddings[i]
        dropout = conv_dropouts[i]

        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=ksize, padding=pad, bias=False))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))

        current_spatial_size = current_spatial_size - ksize + (2 * pad) + 1
        in_channels = out_channels

    flattened_features = in_channels * (current_spatial_size**2)
    layers.append(nn.Flatten())

    # build fc layers
    in_features = flattened_features
    for i in range(n_fc):
        out_features = fc_units[i]
        dropout = fc_dropouts[i]

        layers.append(nn.Linear(in_features, out_features, bias=False))
        layers.append(nn.BatchNorm1d(out_features))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))

        in_features = out_features
        
    layers.append(nn.Linear(in_features, 2 * len(OUT_FEATURES)))

    return nn.Sequential(*layers)


In [31]:
def do_batch(model, data, labels, optimizer, loss_function, train=True):
    data, labels = data.to(DEVICE), labels.to(DEVICE)

    # zero gradients in training
    if train:
        optimizer.zero_grad()

    # get outputs and loss
    outputs = model(data)
    labels = labels.long() # FOR CLASSIFICATION 
    loss = loss_function(outputs, labels)

    # compute gradients and update weights in training
    if train:
        loss.backward()
        optimizer.step()

    return loss, outputs


In [32]:
# TRAINING LOOP FUNCTION    

def objective(trial):
    global BEST_ACC
    model = define_model(trial).to(DEVICE)

    # optimizer and learning rate
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD", "AdamW"])

    kwargs = {"lr": lr}

    if optimizer_name == "AdamW":
        kwargs["weight_decay"] = trial.suggest_float("adamw_weight_decay", 1e-5, 1e-1, log=True)
    elif optimizer_name == "SGD":
        kwargs["momentum"] = trial.suggest_float("sgd_momentum", 0.0, 0.99)

    optimizer = getattr(torch.optim, optimizer_name)(model.parameters(), **kwargs)

    # weights and loss function
    w1 = trial.suggest_float("w1", 1, 3)
    w2 = trial.suggest_float("w2", 1, 3)
    weights = torch.tensor([w1, w2]).to(DEVICE)
    loss_func = nn.CrossEntropyLoss(weight=weights)

    # batch size and loaders
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    if SEED:
        train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=False, drop_last=True)
    else:
         train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)

    # early stopper
    stopper = ut.EarlyStopper(patience=PATIENCE, min_delta=0.0005)

    try:
        for epoch in range(EPOCHS):
        # Training Loop ---------------------------------------------------------------------------------------
            model.train()
            ut.do_epoch(model=model, loader=train_loader, 
                        loss_function=loss_func, device=DEVICE,
                        optimizer=optimizer, train=True)
                
            # Validation Loop --------------------------------------------------------------------------------------
            model.eval()
            with torch.no_grad():
                running_loss, correct = ut.do_epoch(model=model, loader=val_loader, 
                                                loss_function=loss_func, device=DEVICE,
                                                train=False)

                # compute accuracy and report to optuna
                accuracy = correct / len(val_set)
                if accuracy > BEST_ACC:
                    BEST_ACC = float(accuracy)
                trial.report(accuracy, epoch)

                # prune bad trials
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

                # track loss once per epoch to check early stopper
                val_loss = running_loss / len(val_set)
                stopper(val_loss)
                if stopper.early_stop:
                    break

        return accuracy

    # clean up memory after every trial
    finally:
        if "model" in locals():
            del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

## Train Model

In [ ]:
# TRAINS MODEL

if OPTIMIZE:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # use gaussian process
    if SEED:
        sampler = optuna.samplers.GPSampler(seed=SEED)
    else:
        sampler = optuna.samplers.GPSampler()

    study = optuna.create_study(direction="maximize", sampler=sampler,
                                pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
    study.optimize(objective, n_trials=NUM_TRIALS, show_progress_bar=True)

    print("Best trial:")
    trial = study.best_trial
    print(f"  Accuracy: {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
# LOG OPTIMIZATION

if OPTIMIZE:
    logged_params = trial.params.copy()
    logged_params['comment'] = COMMENT
    logged_params['bands'] = BANDS
    logged_params['out_features'] = OUT_FEATURES
    logged_params['seed'] = SEED
    logged_params['epochs'] = EPOCHS
    logged_params['image_size'] = INPUT_SIZE
    logged_params['num_trials'] = NUM_TRIALS
    logged_params['patience'] = PATIENCE
    logged_params['accuracy'] = trial.value
    logged_params['best accuracy'] = BEST_ACC

    logger = ut.RunLogger(filepath=cwd.parent / 'logs' / 'conv.json')
    logger.log_run(logged_params)